# Fit Pink Trombone params to a real speech clip

Gradient descent on the 13-dim parameter trajectory of `pink_trombone_ola` (the faster IIR/OLA FIR synth) to approximate `examples/this-is-samuel.wav`.

Loss: multi-scale log-magnitude STFT L1 (phase-invariant — waveform MSE won't work because glottal pulses won't phase-align with the target).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from scipy.io import wavfile
from scipy.signal import resample_poly
import plotly.graph_objects as go
from IPython.display import Audio, display

from samuel.pink_trombone import (
    pink_trombone_ola,
    PARAM_NAMES,
    N_PARAMS,
    SAMPLE_RATE,
    SAMPLES_PER_FRAME,
    CONTROL_RATE,
)

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Load and resample target audio

In [ ]:
from samuel.data import _load_resampled, load_manifest

MANIFEST_PATH = (Path(".") / "manifests" / "librilight_10h.jsonl").resolve()
CLIP_INDEX = 0          # which file in the manifest to use
START_OFFSET = 1.0      # seconds into the file (skip leading silence)
TARGET_DURATION = 2.0   # seconds

files = load_manifest(MANIFEST_PATH)
df = files[CLIP_INDEX]
print(f"manifest: {len(files)} files")
print(f"using: {df.path.name} ({df.duration:.1f}s @ {df.sample_rate} Hz)")

audio = _load_resampled(df.path, SAMPLE_RATE)  # mono float32, 44100 Hz
start = int(round(START_OFFSET * SAMPLE_RATE))
n_target = int(round(TARGET_DURATION * SAMPLE_RATE))
assert start + n_target <= len(audio), f"file too short: {len(audio)/SAMPLE_RATE:.1f}s"
wav_rs = audio[start : start + n_target].astype(np.float32)

# Truncate to a whole number of control frames
T = wav_rs.shape[0] // SAMPLES_PER_FRAME
T_samples = T * SAMPLES_PER_FRAME
wav_rs = wav_rs[:T_samples]
target = torch.from_numpy(wav_rs).unsqueeze(0).to(device)
print(f"target: shape={tuple(target.shape)}, T={T} frames "
      f"({T/CONTROL_RATE:.2f} s, offset {START_OFFSET:.1f}s)")

print("\nTarget:")
display(Audio(target[0].cpu().numpy(), rate=SAMPLE_RATE))


## 2. Constrained parameterization

Trainable raw scalars for 9 params; vibrato + tractLength are frozen. `to_synth_params` maps raw → constrained [B, T, 13] via sigmoid + affine.

In [ ]:
# (lo, hi) per param. None entries are frozen at their default.
PARAM_SPEC = {
    "frequency":            (80.0, 400.0),
    "voiceness":            (0.0, 1.0),
    "intensity":            (0.0, 1.0),
    "tongueIndex":          (10.0, 35.0),
    "tongueDiameter":       (1.5, 3.5),
    "vibratoWobble":        None,   # frozen at 0
    "vibratoFrequency":     None,   # frozen at 6
    "vibratoGain":          None,   # frozen at 0
    "tractLength":          None,   # frozen at 44
    "constrictionIndex":    (0.0, 44.0),
    "constrictionDiameter": (-0.5, 3.0),
}
FROZEN_VALUES = {
    "vibratoWobble": 0.0,
    "vibratoFrequency": 6.0,
    "vibratoGain": 0.0,
    "tractLength": 44.0,
}

trainable_names = [n for n in PARAM_NAMES if PARAM_SPEC[n] is not None]
n_trainable = len(trainable_names)

# B independent fits launched from different starting points.
B = 64
print(f"trainable params: {n_trainable} x {T} frames x B={B} = {n_trainable * T * B} scalars")

_lo = torch.tensor([PARAM_SPEC[n][0] for n in trainable_names], device=device)
_hi = torch.tensor([PARAM_SPEC[n][1] for n in trainable_names], device=device)

# pyin-derived f0 trajectory. pyin's transition prior breaks at very large
# hops, so we run it at a fine hop and resample to the T control-frame
# timestamps. Unvoiced frames (NaN) are imputed by linear interpolation
# between voiced neighbours (with edge-clamping to the nearest voiced value).
import librosa

FREQ_LO, FREQ_HI = PARAM_SPEC["frequency"]
PYIN_HOP = 512
f0_pyin, _, _ = librosa.pyin(
    target[0].cpu().numpy().astype(np.float32),
    sr=SAMPLE_RATE,
    fmin=FREQ_LO,
    fmax=FREQ_HI,
    frame_length=2048,
    hop_length=PYIN_HOP,
)
t_pyin = librosa.times_like(f0_pyin, sr=SAMPLE_RATE, hop_length=PYIN_HOP)
t_ctrl = (np.arange(T) + 0.5) * SAMPLES_PER_FRAME / SAMPLE_RATE  # frame centres

voiced_mask = np.isfinite(f0_pyin)
if voiced_mask.any():
    f0_init = np.interp(t_ctrl, t_pyin[voiced_mask], f0_pyin[voiced_mask]).astype(np.float32)
else:
    f0_init = np.full(T, 0.5 * (FREQ_LO + FREQ_HI), dtype=np.float32)
f0_init = np.clip(f0_init, FREQ_LO + 1e-3, FREQ_HI - 1e-3)
print(f"pyin: {int(voiced_mask.sum())}/{voiced_mask.size} voiced frames at hop={PYIN_HOP}; "
      f"resampled to T={T} control frames; "
      f"init f0 range [{f0_init.min():.1f}, {f0_init.max():.1f}] Hz")

# Random per-fit starting point for non-frequency params: uniform in normalized
# [0.1, 0.9], constant over T. The frequency column is overridden below with
# the pyin trajectory (same across B).
_gen = torch.Generator(device=device).manual_seed(42)
_init_norm = torch.rand(B, n_trainable, generator=_gen, device=device) * 0.8 + 0.1
_raw_init = torch.log(_init_norm / (1 - _init_norm))  # [B, n_trainable]
raw = _raw_init.unsqueeze(1).expand(B, T, n_trainable).contiguous().clone()

freq_col = trainable_names.index("frequency")
f0_norm = (torch.from_numpy(f0_init).to(device) - FREQ_LO) / (FREQ_HI - FREQ_LO)
f0_norm = f0_norm.clamp(1e-4, 1 - 1e-4)
raw[:, :, freq_col] = torch.log(f0_norm / (1 - f0_norm))  # broadcast over B

raw.requires_grad_(True)

def to_synth_params(raw: torch.Tensor) -> torch.Tensor:
    """raw [B, T, n_trainable] -> synth params [B, T, 13] in valid ranges."""
    B_ = raw.shape[0]
    constrained = torch.sigmoid(raw) * (_hi - _lo) + _lo
    out = torch.zeros(B_, T, N_PARAMS, device=raw.device, dtype=raw.dtype)
    for j, name in enumerate(trainable_names):
        i = PARAM_NAMES.index(name)
        out[..., i] = constrained[..., j]
    for name, val in FROZEN_VALUES.items():
        i = PARAM_NAMES.index(name)
        out[..., i] = val
    return out

# Sanity: print element 0 frame 0
with torch.no_grad():
    p0 = to_synth_params(raw)[0, 0]
    for name in PARAM_NAMES:
        print(f"  {name:22s} = {p0[PARAM_NAMES.index(name)].item():.3f}")


In [ ]:
# plot the pyin f0 trajectory and the initial synth f0 trajectory
with torch.no_grad():
    synth_params = to_synth_params(raw)
    synth_f0 = synth_params[..., PARAM_NAMES.index("frequency")]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_ctrl, y=synth_f0[0].cpu().numpy(), mode="lines", name="init synth f0"))
    fig.add_trace(go.Scatter(x=t_pyin, y=f0_pyin, mode="markers", name="pyin f0"))
    fig.update_layout(title="Initial f0 trajectory from pyin vs synth")
    fig.show()

## 3. Multi-scale STFT loss

In [ ]:
from samuel.losses import MultiScaleLogMagSTFTLoss

# Same loss class the training loop uses (src/samuel/train.py).
loss_fn = MultiScaleLogMagSTFTLoss().to(device)

def per_element_loss(
    pred: torch.Tensor, target: torch.Tensor, loss_fn: MultiScaleLogMagSTFTLoss
) -> torch.Tensor:
    """Same formula as ``loss_fn`` but returns shape [B] instead of a scalar.

    pred [B, S]; target [B, S] or [1, S] (broadcast). ``loss_fn(pred, target.expand(B, -1)).mean()``
    equals ``per_element_loss(pred, target, loss_fn).mean()``.
    """
    losses = pred.new_zeros(pred.shape[0])
    for n_fft in loss_fn.n_ffts:
        window = getattr(loss_fn, f"window_{n_fft}")
        hop = n_fft // loss_fn.hop_div
        s_p = torch.stft(pred,   n_fft=n_fft, hop_length=hop, window=window, return_complex=True).abs()
        s_t = torch.stft(target, n_fft=n_fft, hop_length=hop, window=window, return_complex=True).abs()
        losses = losses + (torch.log1p(s_p) - torch.log1p(s_t)).abs().mean(dim=(1, 2))
    return losses / len(loss_fn.n_ffts)

with torch.no_grad():
    init_pred = pink_trombone_ola(to_synth_params(raw), seed=0)
    init_losses = per_element_loss(init_pred, target, loss_fn)
    # Sanity: per-element mean equals scalar loss_fn over the full batch
    expected = loss_fn(init_pred, target.expand(init_pred.shape[0], -1))
    assert torch.allclose(init_losses.mean(), expected, atol=1e-5)
    print(f"initial losses: min={init_losses.min().item():.4f}  "
          f"max={init_losses.max().item():.4f}  mean={init_losses.mean().item():.4f}")
    print(f"initial pred shape: {tuple(init_pred.shape)}, target shape: {tuple(target.shape)}")


## 4. Training loop

In [ ]:
import time
from tqdm.auto import tqdm

def _norm_np(x: torch.Tensor) -> np.ndarray:
    a = x.detach().cpu().numpy()
    p = np.abs(a).max()
    return a / p * 0.9 if p > 1e-6 else a

def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

N_STEPS = 200
LR = 0.05
IR_LENGTH = 256  # main cost driver: sequential graph depth inside the synth
optimizer = torch.optim.Adam([raw], lr=LR)

# Per-fit best-so-far across steps
best_loss_per_b = torch.full((B,), float("inf"), device=device)
best_pred_per_b = torch.zeros(B, target.shape[1], device=device)
best_params_per_b = torch.zeros(B, T, N_PARAMS, device=device)
loss_history: list[torch.Tensor] = []

phase_t = {"fwd": 0.0, "loss": 0.0, "bwd": 0.0, "opt": 0.0, "display": 0.0}

# Live audio shows the current-best (lowest-loss) element each step.
audio_h = display(
    Audio(np.zeros(SAMPLES_PER_FRAME, dtype=np.float32), rate=SAMPLE_RATE),
    display_id=True,
)

pbar = tqdm(range(N_STEPS), desc="fitting")
for step in pbar:
    _sync(); t0 = time.perf_counter()
    optimizer.zero_grad()
    synth_params = to_synth_params(raw)
    pred = pink_trombone_ola(synth_params, seed=0, ir_length=IR_LENGTH)
    _sync(); t1 = time.perf_counter()

    losses_per = per_element_loss(pred, target, loss_fn)  # [B]
    loss = losses_per.mean()
    _sync(); t2 = time.perf_counter()

    loss.backward()
    _sync(); t3 = time.perf_counter()

    optimizer.step()
    _sync(); t4 = time.perf_counter()

    with torch.no_grad():
        mask = losses_per < best_loss_per_b
        best_loss_per_b = torch.where(mask, losses_per.detach(), best_loss_per_b)
        if mask.any():
            idx = mask.nonzero(as_tuple=True)[0]
            best_pred_per_b[idx]   = pred.detach()[idx]
            best_params_per_b[idx] = synth_params.detach()[idx]
        b_show = int(losses_per.argmin().item())
    audio_h.update(Audio(_norm_np(pred[b_show]), rate=SAMPLE_RATE))
    t5 = time.perf_counter()

    phase_t["fwd"]     += t1 - t0
    phase_t["loss"]    += t2 - t1
    phase_t["bwd"]     += t3 - t2
    phase_t["opt"]     += t4 - t3
    phase_t["display"] += t5 - t4

    loss_history.append(losses_per.detach().clone())
    pbar.set_postfix(
        mean=f"{losses_per.mean().item():.4f}",
        best=f"{best_loss_per_b.min().item():.4f}",
        worst=f"{best_loss_per_b.max().item():.4f}",
        step_s=f"{t4 - t0:.2f}",
    )

loss_history_t = torch.stack(loss_history).cpu().numpy()  # [N_STEPS, B]

best_idx  = int(best_loss_per_b.argmin().item())
worst_idx = int(best_loss_per_b.argmax().item())
best_pred   = best_pred_per_b[best_idx:best_idx+1]
best_params = best_params_per_b[best_idx:best_idx+1]

print(f"\nbest-over-steps loss: min={best_loss_per_b.min().item():.4f}  "
      f"max={best_loss_per_b.max().item():.4f}  mean={best_loss_per_b.mean().item():.4f}")
print(f"best fit:  element {best_idx}  loss {best_loss_per_b[best_idx].item():.4f}")
print(f"worst fit: element {worst_idx}  loss {best_loss_per_b[worst_idx].item():.4f}\n")

total = sum(phase_t.values())
print(f"{'phase':<10} {'total (s)':>10} {'ms/step':>10} {'%':>6}")
for k, v in phase_t.items():
    print(f"{k:<10} {v:10.2f} {v / N_STEPS * 1000:10.1f} {v / total * 100:5.1f}%")
print(f"{'TOTAL':<10} {total:10.2f} {total / N_STEPS * 1000:10.1f} {100.0:5.1f}%")


## 5. Loss curve (plotly)

In [ ]:
fig = go.Figure()
for b in range(B):
    fig.add_trace(go.Scatter(
        y=loss_history_t[:, b],
        mode="lines",
        line=dict(width=1, color="rgba(120,120,120,0.4)"),
        showlegend=False,
        hoverinfo="skip",
    ))
fig.add_trace(go.Scatter(y=loss_history_t[:, best_idx],  mode="lines",
                         line=dict(width=2.5, color="green"),
                         name=f"best  (b={best_idx}, L={best_loss_per_b[best_idx].item():.3f})"))
fig.add_trace(go.Scatter(y=loss_history_t[:, worst_idx], mode="lines",
                         line=dict(width=2.5, color="red"),
                         name=f"worst (b={worst_idx}, L={best_loss_per_b[worst_idx].item():.3f})"))
fig.update_layout(
    title=f"Fitting loss vs. step ({B} parallel fits, different starting points)",
    xaxis_title="step",
    yaxis_title="loss (log scale)",
    yaxis_type="log",
    template="plotly_white",
    width=900,
    height=420,
)
fig.show()


## 5b. Distribution of best-over-steps loss

Each of the B fits is independent (different random starting point, same target). If different starts converge to similar loss, the loss surface has one dominant basin around this scale. If they spread out, multiple local optima.

In [ ]:
final_losses = best_loss_per_b.cpu().numpy()
order = np.argsort(final_losses)

from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=("histogram", "sorted per-fit losses"))
fig.add_trace(go.Histogram(x=final_losses, nbinsx=10, showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(B), y=final_losses[order],
                         mode="markers+lines", marker=dict(size=8),
                         showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="loss",  row=1, col=1)
fig.update_xaxes(title_text="rank",  row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="loss",  row=1, col=2)
fig.update_layout(template="plotly_white", width=900, height=350,
                  title=f"Best-over-steps loss across {B} fits")
fig.show()
print(f"min={final_losses.min():.4f}  max={final_losses.max():.4f}  "
      f"mean={final_losses.mean():.4f}  std={final_losses.std():.4f}  "
      f"max/min ratio={final_losses.max()/final_losses.min():.2f}x")


## 6. Listen

In [ ]:
def norm(x: torch.Tensor) -> np.ndarray:
    x = x.cpu().numpy()
    peak = np.abs(x).max()
    return x / peak * 0.9 if peak > 1e-6 else x

print("Target:")
display(Audio(norm(target[0]), rate=SAMPLE_RATE))

print(f"Best fit (element {best_idx}, loss {best_loss_per_b[best_idx].item():.4f}):")
display(Audio(norm(best_pred_per_b[best_idx]), rate=SAMPLE_RATE))

print(f"Worst fit (element {worst_idx}, loss {best_loss_per_b[worst_idx].item():.4f}):")
display(Audio(norm(best_pred_per_b[worst_idx]), rate=SAMPLE_RATE))


## 7. Fitted parameter trajectories

In [ ]:
show_names = ["frequency", "voiceness", "tongueIndex", "tongueDiameter",
              "constrictionIndex", "constrictionDiameter"]
frames = np.arange(T) / CONTROL_RATE  # seconds

fig = go.Figure()
for name in show_names:
    i = PARAM_NAMES.index(name)
    lo, hi = PARAM_SPEC[name]
    # Invert the rescaling: sigmoid(raw) = (constrained - lo) / (hi - lo) in [0, 1].
    sig = (best_params[0, :, i].cpu().numpy() - lo) / (hi - lo)
    fig.add_trace(go.Scatter(x=frames, y=sig, mode="lines+markers", name=name))
fig.update_layout(
    title="Fitted parameter trajectories — sigmoid(raw), shared [0, 1] axis",
    xaxis_title="time (s)",
    yaxis_title="sigmoid(raw) ∈ [0, 1]",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    width=900,
    height=450,
)
fig.show()


## 8. Spectrum comparison (target vs. fitted OLA)

In [ ]:
import librosa
from plotly.subplots import make_subplots

N_FFT = 2048
HOP = N_FFT // 4
N_MELS = 80
F_MIN, F_MAX = 0.0, 2000.0
_win = torch.hann_window(N_FFT, device=device)

_mel_fb = torch.from_numpy(
    librosa.filters.mel(sr=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX)
).to(device)  # [N_MELS, F]

mel_centers = librosa.mel_frequencies(n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX)

def _logmel(x: torch.Tensor) -> np.ndarray:
    """[T_samples] -> log-mel spectrogram [N_MELS, N_frames] on CPU."""
    s = torch.stft(x.unsqueeze(0), N_FFT, HOP, window=_win, return_complex=True).abs()  # [1, F, N]
    m = _mel_fb @ s[0]  # [N_MELS, N]
    return torch.log1p(m).cpu().numpy()

mel_target = _logmel(target[0])
mel_fit    = _logmel(best_pred[0])

times = np.arange(mel_target.shape[1]) * HOP / SAMPLE_RATE
zmax = float(max(mel_target.max(), mel_fit.max()))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("target", "fitted (OLA)"))
for col, spec in [(1, mel_target), (2, mel_fit)]:
    fig.add_trace(go.Heatmap(z=spec, x=times, y=mel_centers, zmin=0, zmax=zmax,
                             colorscale="Viridis", showscale=(col == 2)),
                  row=1, col=col)
fig.update_xaxes(title_text="time (s)")
fig.update_yaxes(title_text="mel center freq (Hz)", row=1, col=1)
fig.update_layout(title=f"log-mel spectrogram (n_mels={N_MELS}, n_fft={N_FFT})",
                  template="plotly_white", width=1000, height=400)
fig.show()

# Time-averaged mel spectra for a direct line-plot comparison
avg_target = mel_target.mean(axis=1)
avg_fit    = mel_fit.mean(axis=1)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_target, name="target",       mode="lines"))
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_fit,    name="fitted (OLA)", mode="lines"))
fig2.update_layout(
    title="Time-averaged log-mel spectrum",
    xaxis_title="mel center freq (Hz)", yaxis_title="log(1 + mel energy)",
    template="plotly_white", width=900, height=400,
)
fig2.show()

## 9. Resynth with exact (non-OLA) waveguide

Fitting used `pink_trombone_ola` (OLA FIR approximation). Here we resynthesize the same best params through the exact sequential Kelly–Lochbaum waveguide (`pink_trombone`) to see how much the OLA approximation is shaping the result.

In [ ]:
from samuel.pink_trombone import pink_trombone

with torch.no_grad():
    pred_exact = pink_trombone(best_params, seed=0)
    loss_ola   = multi_stft_loss(best_pred,  target).item()
    loss_exact = multi_stft_loss(pred_exact, target).item()

print(f"multi-STFT loss vs. target:")
print(f"  OLA   (training synth): {loss_ola:.4f}")
print(f"  exact (sequential WG):  {loss_exact:.4f}")

rms_diff = (pred_exact - best_pred).pow(2).mean().sqrt().item()
rms_exact = pred_exact.pow(2).mean().sqrt().item()
print(f"  RMS(exact - OLA) = {rms_diff:.4f}   (relative to exact RMS {rms_exact:.4f}: {rms_diff/rms_exact:.2%})")

print("\nTarget:")
display(Audio(norm(target[0]),     rate=SAMPLE_RATE))
print("Fitted — OLA FIR (what the optimizer saw):")
display(Audio(norm(best_pred[0]),  rate=SAMPLE_RATE))
print("Fitted — exact sequential waveguide:")
display(Audio(norm(pred_exact[0]), rate=SAMPLE_RATE))

# Log-mel spectrograms: exact vs OLA (same params)
mel_exact = _logmel(pred_exact[0])
zmax2 = float(max(mel_fit.max(), mel_exact.max()))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("fitted (OLA)", "fitted (exact)"))
for col, spec in [(1, mel_fit), (2, mel_exact)]:
    fig.add_trace(go.Heatmap(z=spec, x=times, y=mel_centers, zmin=0, zmax=zmax2,
                             colorscale="Viridis", showscale=(col == 2)),
                  row=1, col=col)
fig.update_xaxes(title_text="time (s)")
fig.update_yaxes(title_text="mel center freq (Hz)", row=1, col=1)
fig.update_layout(title="OLA vs. exact waveguide (same fitted params) — log-mel",
                  template="plotly_white", width=1000, height=400)
fig.show()

avg_exact = mel_exact.mean(axis=1)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_target, name="target",         mode="lines"))
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_fit,    name="fitted (OLA)",   mode="lines"))
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_exact,  name="fitted (exact)", mode="lines"))
fig2.update_layout(
    title="Time-averaged log-mel spectrum — target vs OLA vs exact",
    xaxis_title="mel center freq (Hz)", yaxis_title="log(1 + mel energy)",
    template="plotly_white", width=900, height=400,
)
fig2.show()

## 10. Pitch (f0) comparison

Estimated with `librosa.pyin` (probabilistic YIN) — robust to voiced/unvoiced transitions, NaN on unvoiced frames. Compared against the commanded `frequency` parameter the optimizer settled on.

In [ ]:
PITCH_FMIN, PITCH_FMAX = 70.0, 500.0
PITCH_HOP = 512  # ~11.6 ms at 44.1 kHz

def _pyin_f0(x: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    wav = x.detach().cpu().numpy().astype(np.float32)
    f0, _, _ = librosa.pyin(
        wav, sr=SAMPLE_RATE,
        fmin=PITCH_FMIN, fmax=PITCH_FMAX,
        frame_length=2048, hop_length=PITCH_HOP,
    )
    t = librosa.times_like(f0, sr=SAMPLE_RATE, hop_length=PITCH_HOP)
    return t, f0

t_pit, f0_target = _pyin_f0(target[0])
_,     f0_ola    = _pyin_f0(best_pred[0])
_,     f0_exact  = _pyin_f0(pred_exact[0])

freq_idx = PARAM_NAMES.index("frequency")
commanded = best_params[0, :, freq_idx].cpu().numpy()
t_cmd = np.arange(T) / CONTROL_RATE

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_pit, y=f0_target, name="target (pyin)",       mode="lines"))
fig.add_trace(go.Scatter(x=t_pit, y=f0_ola,    name="fitted OLA (pyin)",   mode="lines"))
fig.add_trace(go.Scatter(x=t_pit, y=f0_exact,  name="fitted exact (pyin)", mode="lines"))
fig.add_trace(go.Scatter(x=t_cmd, y=commanded, name="commanded frequency", mode="lines+markers",
                         line=dict(dash="dash")))
fig.update_layout(
    title="Pitch (f0) trajectories",
    xaxis_title="time (s)",
    yaxis_title="f0 (Hz)",
    yaxis_range=[PITCH_FMIN, PITCH_FMAX],
    template="plotly_white", width=1000, height=450,
)
fig.show()

# MAE in cents on frames where both target and each synth are voiced
def _cents_mae(a: np.ndarray, b: np.ndarray) -> float:
    m = np.isfinite(a) & np.isfinite(b)
    if not m.any():
        return float("nan")
    return float(np.mean(np.abs(1200.0 * np.log2(a[m] / b[m]))))

print(f"voiced-frame MAE vs. target:")
print(f"  fitted OLA:   {_cents_mae(f0_target, f0_ola):.1f} cents")
print(f"  fitted exact: {_cents_mae(f0_target, f0_exact):.1f} cents")